# 🌱 SmartPlant IoT 2.0 — Irrigação multizona

**Trabalho final de IoT | Google Colab + Paho MQTT + MQTTX**

A plantação é dividida em **6 zonas independentes**. Cada zona possui umidade, temperatura, reservatório e bomba **virtuais**, e seus próprios limites de irrigação. Um cliente MQTT central reúne os dados e publica um **relatório geral** para o MQTTX. Também gera CSV e gráficos.

**Como usar:** execute as células na ordem. Para receber os dados no MQTTX, mantenha a célula de simulação rodando. O campo `timestamp_utc` foi removido das mensagens MQTT.

> Broker público: não envie informações pessoais nem acione equipamentos reais.

## 1. Instalar as bibliotecas

Execute uma vez por sessão do Colab.

In [1]:
%pip -q install "paho-mqtt>=2,<3" pandas matplotlib

## 2. Definir os sensores virtuais, o controle por zona e o agregador MQTT

Esta célula contém toda a implementação, sem precisar enviar arquivos `.py` adicionais.

In [4]:
from __future__ import annotations

import json
import random
import threading
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

BROKER = "broker.emqx.io"
PORTA = 1883
GRUPO = "colabba66c73d"    # mesmo grupo usado anteriormente no MQTTX
INTERVALO = 2.0
N_AMOSTRAS = 120          # aproximadamente 4 minutos
INTERVALO_RELATORIO = 10.0
LIMITE_IRRIGAR = 35.0
LIMITE_PARAR = 60.0
NIVEL_MINIMO = 10.0

# Cada zona tem condicoes iniciais diferentes, inclusive uma quase sem agua.
CONFIGURACOES_ZONAS = {
    "zona01": {"umidade": 23.0, "temperatura": 26.0, "reservatorio": 84.0, "limite_ligar": 35, "limite_desligar": 60},
    "zona02": {"umidade": 49.0, "temperatura": 27.0, "reservatorio": 72.0, "limite_ligar": 39, "limite_desligar": 64},
    "zona03": {"umidade": 68.0, "temperatura": 25.0, "reservatorio": 91.0, "limite_ligar": 31, "limite_desligar": 56},
    "zona04": {"umidade": 26.0, "temperatura": 29.0, "reservatorio":  8.0, "limite_ligar": 36, "limite_desligar": 61},
    "zona05": {"umidade": 37.0, "temperatura": 24.0, "reservatorio": 61.0, "limite_ligar": 40, "limite_desligar": 66},
    "zona06": {"umidade": 73.0, "temperatura": 25.0, "reservatorio": 96.0, "limite_ligar": 33, "limite_desligar": 59},
}


def limitar(valor, minimo=0.0, maximo=100.0):
    return max(minimo, min(maximo, float(valor)))


class ZonaIrrigacao:
    """Uma area fisicamente/logicamente independente, com propria histerese."""
    def __init__(self, identificador, configuracao):
        self.id = identificador
        self.lock = threading.RLock()
        self.sorteio = random.Random(identificador)  # aleatorio reprodutivel por zona
        self.umidade = float(configuracao["umidade"])
        self.temperatura = float(configuracao["temperatura"])
        self.reservatorio = float(configuracao["reservatorio"])
        self.limite_ligar = float(configuracao.get("limite_ligar", LIMITE_IRRIGAR))
        self.limite_desligar = float(configuracao.get("limite_desligar", LIMITE_PARAR))
        self.modo = "auto"
        self.bomba = False
        self.ciclos = 0
        self.amostras = 0

    def _controlar(self):
        antes = self.bomba
        if self.reservatorio <= NIVEL_MINIMO:
            self.bomba = False
        elif self.modo == "auto":
            if self.umidade <= self.limite_ligar:
                self.bomba = True
            elif self.umidade >= self.limite_desligar:
                self.bomba = False
        if self.bomba and not antes:
            self.ciclos += 1

    def aplicar_comando(self, comando):
        """Aceita comandos JSON da zona e protege contra valores invalidos."""
        if not isinstance(comando, dict) or not comando:
            raise ValueError("Envie um objeto JSON nao vazio.")
        permitidos = {"modo", "bomba", "umidade_solo", "nivel_reservatorio", "limite_ligar", "limite_desligar"}
        desconhecidos = set(comando) - permitidos
        if desconhecidos:
            raise ValueError("Campos desconhecidos: " + ", ".join(sorted(desconhecidos)))

        with self.lock:
            modo = comando.get("modo", self.modo)
            if modo not in {"auto", "manual"}:
                raise ValueError("modo deve ser 'auto' ou 'manual'.")
            if "bomba" in comando:
                if modo != "manual" or comando["bomba"] not in {"on", "off"}:
                    raise ValueError("bomba requer modo manual e valor 'on' ou 'off'.")
            for chave in ("umidade_solo", "nivel_reservatorio"):
                if chave in comando:
                    valor = comando[chave]
                    if (isinstance(valor, bool) or not isinstance(valor, (int, float))
                            or not 0 <= valor <= 100):
                        raise ValueError(chave + " deve ser um numero entre 0 e 100.")

            novo_ligar = comando.get("limite_ligar", self.limite_ligar)
            novo_desligar = comando.get("limite_desligar", self.limite_desligar)
            if (any(isinstance(x, bool) or not isinstance(x, (int, float)) for x in (novo_ligar, novo_desligar))
                    or not 0 <= novo_ligar < novo_desligar <= 100):
                raise ValueError("Os limites devem satisfazer 0 <= limite_ligar < limite_desligar <= 100.")

            self.limite_ligar = float(novo_ligar)
            self.limite_desligar = float(novo_desligar)
            if "modo" in comando:
                self.modo = modo
                self.bomba = False  # troca de modo nunca preserva atuacao anterior
            if "umidade_solo" in comando:
                self.umidade = float(comando["umidade_solo"])
            if "nivel_reservatorio" in comando:
                self.reservatorio = float(comando["nivel_reservatorio"])
            if "bomba" in comando:
                self.bomba = comando["bomba"] == "on"
            self._controlar()
            if comando.get("bomba") == "on" and not self.bomba:
                return "Bomba bloqueada: nivel de agua insuficiente."
            return "Comando aceito para " + self.id

    def passo(self):
        """Evolui exclusivamente o proprio ambiente desta zona."""
        with self.lock:
            self._controlar()
            self.temperatura = limitar(
                self.temperatura + self.sorteio.uniform(-0.35, 0.35), 17, 39
            )
            if self.bomba:
                self.umidade = limitar(self.umidade + self.sorteio.uniform(3.7, 5.3))
                self.reservatorio = limitar(self.reservatorio - self.sorteio.uniform(1.2, 2.2))
            else:
                self.umidade = limitar(self.umidade - self.sorteio.uniform(0.8, 1.6))
            self._controlar()
            self.amostras += 1
            if self.reservatorio <= NIVEL_MINIMO:
                alerta = "RESERVATORIO_BAIXO"
            elif self.bomba:
                alerta = "IRRIGANDO"
            elif self.umidade <= self.limite_ligar:
                alerta = "SOLO_SECO"
            else:
                alerta = "NORMAL"
            return {
                "zona": self.id,
                "umidade_solo": round(self.umidade, 1),
                "temperatura_c": round(self.temperatura, 1),
                "nivel_reservatorio": round(self.reservatorio, 1),
                "bomba": int(self.bomba),
                "modo": self.modo,
                "limite_ligar": self.limite_ligar,
                "limite_desligar": self.limite_desligar,
                "alerta": alerta,
                "ciclos_irrigacao": self.ciclos,
                "amostras": self.amostras,
            }


class CentralRelatorios:
    """Cliente MQTT central, agregador e gravador de resultados."""
    def __init__(self, base, total_zonas):
        self.base = base
        self.total_zonas = total_zonas
        self.lock = threading.RLock()
        self.online = threading.Event()
        self.cliente = None
        self.historico = []
        self.ultimos = {}
        self.chaves_recebidas = set()
        self.relatorios_publicados = 0

    def _guardar(self, dados, origem):
        if not isinstance(dados, dict) or "zona" not in dados or "amostras" not in dados:
            return False
        zona = str(dados["zona"])
        if zona not in self.total_zonas:
            return False
        try:
            amostra = int(dados["amostras"])
        except (TypeError, ValueError):
            return False
        chave = (zona, amostra)
        with self.lock:
            if chave in self.chaves_recebidas:
                return False
            self.chaves_recebidas.add(chave)
            registro = dict(dados)
            # Data/hora apenas no CSV, nunca no JSON publicado no MQTTX.
            registro["recebido_em_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
            registro["origem"] = origem
            self.historico.append(registro)
            self.ultimos[zona] = dict(dados)
        return True

    def receber_local(self, dados):
        return self._guardar(dados, "simulacao_local_offline")

    def conectar(self):
        import paho.mqtt.client as mqtt
        cliente = mqtt.Client(
            callback_api_version=mqtt.CallbackAPIVersion.VERSION2,
            client_id="smartplant-central-" + uuid.uuid4().hex[:7],
        )

        def conectado(client, userdata, flags, reason_code, properties):
            if reason_code == 0:
                self.online.set()
                client.subscribe(self.base + "/zonas/+/telemetria", qos=1)
                print("[MQTT] Central conectada e ouvindo todas as zonas.")
            else:
                print("[MQTT] Central: conexao recusada:", reason_code)

        def desconectado(client, userdata, disconnect_flags, reason_code, properties):
            self.online.clear()

        def mensagem(client, userdata, msg):
            try:
                dados = json.loads(msg.payload.decode("utf-8"))
                zona_no_topico = msg.topic.split("/")[-2]
                if dados.get("zona") == zona_no_topico:
                    self._guardar(dados, "broker_mqtt")
            except (ValueError, AttributeError, UnicodeError, TypeError):
                pass

        cliente.on_connect = conectado
        cliente.on_disconnect = desconectado
        cliente.on_message = mensagem
        self.cliente = cliente
        cliente.connect_async(BROKER, PORTA, keepalive=45)
        cliente.loop_start()
        return self.online.wait(timeout=8)

    def geral(self):
        with self.lock:
            ultimos = [dict(v) for _, v in sorted(self.ultimos.items())]
            total = len(self.historico)
        return {
            "zonas_previstas": len(self.total_zonas),
            "zonas_com_dados": len(ultimos),
            "umidade_media": round(sum(x["umidade_solo"] for x in ultimos) / len(ultimos), 1) if ultimos else None,
            "bombas_ligadas": sum(x["bomba"] for x in ultimos),
            "zonas_com_alerta": [x["zona"] for x in ultimos if x["alerta"] not in ("NORMAL", "IRRIGANDO")],
            "amostras_recebidas": total,
            "situacao": "SIMULACAO",
            "zonas": [
                {"zona": x["zona"], "umidade": x["umidade_solo"],
                 "bomba": x["bomba"], "limite_ligar": x["limite_ligar"],
                 "limite_desligar": x["limite_desligar"], "alerta": x["alerta"]}
                for x in ultimos
            ],
        }

    def publicar_resumo(self):
        resumo = self.geral()
        if self.online.is_set() and self.cliente is not None:
            resultado = self.cliente.publish(
                self.base + "/relatorio/geral", json.dumps(resumo, ensure_ascii=False), qos=1
            )
            if resultado.rc == 0:
                self.relatorios_publicados += 1
        print(
            "[RELATORIO] zonas={}/{} | solo medio={}% | bombas={} | alertas={} | leituras={}".format(
                resumo["zonas_com_dados"], resumo["zonas_previstas"], resumo["umidade_media"],
                resumo["bombas_ligadas"], ",".join(resumo["zonas_com_alerta"]) or "nenhum",
                resumo["amostras_recebidas"],
            )
        )
        return resumo

    def tabelas(self):
        with self.lock:
            historico = list(self.historico)
        dados = pd.DataFrame(historico)
        if dados.empty:
            return dados, pd.DataFrame()
        linhas = []
        for zona, parte in dados.sort_values("amostras").groupby("zona", sort=True):
            fim = parte.iloc[-1]
            linhas.append({
                "zona": zona,
                "leituras": len(parte),
                "umidade_media_pct": round(float(parte["umidade_solo"].mean()), 1),
                "umidade_final_pct": float(fim["umidade_solo"]),
                "temperatura_media_c": round(float(parte["temperatura_c"].mean()), 1),
                "reservatorio_final_pct": float(fim["nivel_reservatorio"]),
                "tempo_irrigando_pct": round(100 * float(parte["bomba"].mean()), 1),
                "ciclos_irrigacao": int(fim["ciclos_irrigacao"]),
                "alertas_observados": int(parte["alerta"].isin(["SOLO_SECO", "RESERVATORIO_BAIXO"]).sum()),
                "modo_final": fim["modo"],
                "limite_ligar": fim["limite_ligar"],
                "limite_desligar": fim["limite_desligar"],
                "situacao_final": fim["alerta"],
            })
        return dados, pd.DataFrame(linhas)

    def encerrar(self):
        if self.cliente is not None:
            try:
                self.cliente.disconnect()
                self.cliente.loop_stop()
            except Exception:
                pass
            self.cliente = None
        self.online.clear()


class TrabalhadorZona:
    """Cada zona roda em uma thread e possui sua propria conexao MQTT."""
    def __init__(self, zona, central, base, n, intervalo, parar, usar_mqtt):
        self.zona = zona
        self.central = central
        self.base = base
        self.n = n
        self.intervalo = intervalo
        self.parar = parar
        self.usar_mqtt = usar_mqtt
        self.online = threading.Event()
        self.cliente = None
        self.thread = None
        self._bomba_anterior = None

    def conectar(self):
        if not self.usar_mqtt:
            return
        import paho.mqtt.client as mqtt
        cliente = mqtt.Client(
            callback_api_version=mqtt.CallbackAPIVersion.VERSION2,
            client_id="sp-" + self.zona.id + "-" + uuid.uuid4().hex[:7],
        )

        def conectado(client, userdata, flags, reason_code, properties):
            if reason_code == 0:
                self.online.set()
                client.subscribe(self.base + "/zonas/" + self.zona.id + "/comando", qos=1)
                print("[MQTT]", self.zona.id, "conectada; pronta para comandos.")
            else:
                print("[MQTT]", self.zona.id, "conexao recusada:", reason_code)

        def desconectado(client, userdata, disconnect_flags, reason_code, properties):
            self.online.clear()

        def mensagem(client, userdata, msg):
            try:
                dados = json.loads(msg.payload.decode("utf-8"))
                resposta = self.zona.aplicar_comando(dados)
                ok = True
            except (ValueError, TypeError, UnicodeError, AttributeError) as erro:
                resposta, ok = str(erro), False
            client.publish(
                self.base + "/zonas/" + self.zona.id + "/resposta",
                json.dumps({"zona": self.zona.id, "aceito": ok, "mensagem": resposta}, ensure_ascii=False),
                qos=1,
            )
            print("[COMANDO]", self.zona.id, "-", resposta)

        cliente.on_connect = conectado
        cliente.on_disconnect = desconectado
        cliente.on_message = mensagem
        self.cliente = cliente
        cliente.connect_async(BROKER, PORTA, keepalive=45)
        cliente.loop_start()

    def publicar(self, dado):
        if self.cliente is not None and self.online.is_set() and self.central.online.is_set():
            resultado = self.cliente.publish(
                self.base + "/zonas/" + self.zona.id + "/telemetria",
                json.dumps(dado, ensure_ascii=False), qos=1,
            )
            if resultado.rc == 0:
                return
        self.central.receber_local(dado)  # fallback claramente identificado no CSV

    def executar(self):
        for i in range(self.n):
            if self.parar.is_set():
                break
            dado = self.zona.passo()
            self.publicar(dado)
            if i == 0 or (i + 1) % 15 == 0 or dado["bomba"] != self._bomba_anterior:
                print(
                    "[{}] {:03d} | solo={:5.1f}% | agua={:5.1f}% | bomba={} | {}".format(
                        self.zona.id, dado["amostras"], dado["umidade_solo"],
                        dado["nivel_reservatorio"], dado["bomba"], dado["alerta"]
                    )
                )
            self._bomba_anterior = dado["bomba"]
            if i < self.n - 1 and self.parar.wait(self.intervalo):
                break

    def iniciar(self):
        self.thread = threading.Thread(target=self.executar, name=self.zona.id, daemon=True)
        self.thread.start()

    def encerrar(self):
        if self.cliente is not None:
            try:
                self.cliente.disconnect()
                self.cliente.loop_stop()
            except Exception:
                pass
            self.cliente = None
        self.online.clear()


def executar_multizona(
    n=N_AMOSTRAS,
    intervalo=INTERVALO,
    usar_mqtt=True,
    grupo=GRUPO,
    numero_zonas=6,
    salvar_pasta=".",
    intervalo_relatorio=INTERVALO_RELATORIO,
):
    if not 1 <= numero_zonas <= len(CONFIGURACOES_ZONAS):
        raise ValueError("numero_zonas precisa estar entre 1 e 6.")
    if n < 1 or intervalo < 0 or intervalo_relatorio <= 0:
        raise ValueError("Use n >= 1, intervalo >= 0 e intervalo_relatorio > 0.")

    selecionadas = dict(list(CONFIGURACOES_ZONAS.items())[:numero_zonas])
    base = "inatel/smartplant/" + grupo
    print("\n=== SmartPlant IoT 2.0 - MULTIZONA ===")
    print("Grupo:", grupo, "| Zonas:", ", ".join(selecionadas))
    print("MQTTX - ASSINAR:", base + "/#")
    print("MQTTX - RELATORIO:", base + "/relatorio/geral")
    print("MQTTX - COMANDAR UMA ZONA:", base + "/zonas/zona02/comando")
    print('Exemplo: {"modo":"auto","umidade_solo":20}')
    print("Broker publico:", BROKER, PORTA)

    central = CentralRelatorios(base, selecionadas)
    mqtt_ativo = False
    if usar_mqtt:
        try:
            mqtt_ativo = central.conectar()
            if not mqtt_ativo:
                print("[AVISO] MQTT indisponivel neste Colab; executando offline e gerando CSV.")
                central.encerrar()
        except Exception as erro:
            print("[AVISO] Falha ao iniciar MQTT:", erro, "| Usando modo offline.")
            central.encerrar()

    parar = threading.Event()
    trabalhadores = [
        TrabalhadorZona(ZonaIrrigacao(nome, cfg), central, base, int(n), float(intervalo), parar, mqtt_ativo)
        for nome, cfg in selecionadas.items()
    ]
    try:
        if mqtt_ativo:
            for t in trabalhadores:
                t.conectar()
            # Aguarda ate 8 segundos no total, nao 8 segundos por zona.
            fim = time.monotonic() + 8
            for t in trabalhadores:
                if not t.online.wait(timeout=max(0, fim - time.monotonic())):
                    print("[AVISO]", t.zona.id, "sem conexao MQTT; telemetria local alternativa.")
        for t in trabalhadores:
            t.iniciar()

        proximo = time.monotonic() + intervalo_relatorio
        while any(t.thread.is_alive() for t in trabalhadores):
            if time.monotonic() >= proximo:
                central.publicar_resumo()
                proximo = time.monotonic() + intervalo_relatorio
            time.sleep(0.15)
        time.sleep(0.6) if mqtt_ativo else None  # permite receber ultimas PUBs
        central.publicar_resumo()
    except KeyboardInterrupt:
        print("\n[PARADA] Interrompido pelo usuario; salvando os dados recebidos.")
    finally:
        parar.set()
        for t in trabalhadores:
            if t.thread is not None:
                t.thread.join(timeout=3)
            t.encerrar()
        central.encerrar()

    df, relatorio = central.tabelas()
    destino = Path(salvar_pasta)
    destino.mkdir(parents=True, exist_ok=True)
    csv_dados = destino / "smartplant_multizona_telemetria.csv"
    csv_relatorio = destino / "smartplant_multizona_relatorio.csv"
    df.to_csv(csv_dados, index=False)
    relatorio.to_csv(csv_relatorio, index=False)
    print("\nArquivos salvos:")
    print(" -", csv_dados.resolve())
    print(" -", csv_relatorio.resolve())
    print("\nRELATORIO FINAL POR ZONA:")
    if not relatorio.empty:
        print(relatorio.to_string(index=False))
    else:
        print("Nenhuma leitura coletada.")
    return df, relatorio


def graficos_multizona(df):
    """Dois graficos simples para os slides e a demonstracao."""
    import matplotlib.pyplot as plt
    if df.empty:
        print("Sem dados para graficos.")
        return
    # Um grafico por tipo: evita misturar unidade de percentual com liga/desliga.
    fig, ax = plt.subplots(figsize=(11, 5))
    for nome, parte in df.groupby("zona"):
        parte = parte.sort_values("amostras")
        ax.plot(parte["amostras"], parte["umidade_solo"], label=nome)
    ax.axhline(LIMITE_IRRIGAR, ls="--", alpha=.6, label="liga <=35%")
    ax.axhline(LIMITE_PARAR, ls=":", alpha=.6, label="desliga >=60%")
    ax.set(title="Umidade do solo por zona - SmartPlant IoT", xlabel="Amostra", ylabel="Umidade (%)", ylim=(0, 100))
    ax.legend(ncol=4)
    ax.grid(alpha=.25)
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(11, 4.4))
    for i, (nome, parte) in enumerate(sorted(df.groupby("zona"))):
        parte = parte.sort_values("amostras")
        ax.step(parte["amostras"], parte["bomba"] + i * 1.4, where="post", label=nome)
    ax.set(title="Irrigacao independente - 0=desligada / 1=ligada", xlabel="Amostra", ylabel="Zona (faixas separadas)")
    ax.set_yticks([i * 1.4 + .5 for i in range(df["zona"].nunique())])
    ax.set_yticklabels([nome for nome, _ in sorted(df.groupby("zona"))])
    ax.grid(alpha=.25)
    fig.tight_layout()
    plt.show()


## 3. Executar a simulação



In [ ]:
GRUPO = "colabba66c73d"  # identico ao grupo que voce ja usa no MQTTX

dados, relatorio = executar_multizona(
    grupo=GRUPO,
    numero_zonas=6,   # 1 a 6 zonas
    n=600,            # numero de amostras por zona
    intervalo=2,      # segundos entre amostras de cada zona
    intervalo_relatorio=10,
    usar_mqtt=True,
)


## 4. Mostrar os relatórios e gráficos após a execução

In [ ]:
from IPython.display import display

print("RESUMO POR ZONA")
display(relatorio)
print("ULTIMAS AMOSTRAS")
display(dados.sort_values(["zona", "amostras"]).groupby("zona").tail(1))
graficos_multizona(dados)


## 5. Download opcional dos arquivos CSV

In [ ]:
from google.colab import files
files.download("smartplant_multizona_telemetria.csv")
files.download("smartplant_multizona_relatorio.csv")